# Practical 16 — Text-to-Signal ML Pipeline

**Large Language Models in Finance · Chapter 16 — Artificial Intelligence, Machine
Learning, and Text in Finance**

This notebook runs the chapter's *text-as-data* pipeline end to end on a bundled
corpus of labelled financial sentences: each sentence is turned into a **TF-IDF**
feature vector, a **logistic-regression** classifier (trained by plain gradient
descent in NumPy) learns the up (1) / down (0) signal, and we report the
**out-of-sample accuracy** on a held-out split.

In the real project you'd open the folder in Claude Code / Cline and let an agent
orchestrate the deterministic tools in `tools/`. Colab can't run that agentic loop,
so this notebook is the Colab-friendly counterpart: it calls the **exact same
reference tools directly**, fully offline, so you see the whole pipeline run and its
real outputs.

In [ ]:
# Colab: run once to install this practical's dependencies.
# (Locally, skip if you ran `pip install -r requirements.txt`.)
%pip install numpy

In [ ]:
# --- setup: make this practical's tools importable (works locally & in Colab) ---
import os, sys, pathlib, subprocess
PRACTICAL = "16-ai-ml-finance-text"
def _locate():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if base.name == PRACTICAL and (base / "tools").is_dir():
            return base
        cand = base / "code" / "practicals" / PRACTICAL
        if cand.is_dir():
            return cand
    return None
root = _locate()
if root is None:
    if not pathlib.Path("llm-finance-book").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jfimbett/llm-finance-book.git"], check=True)
    root = pathlib.Path("llm-finance-book/code/practicals") / PRACTICAL
root = root.resolve()
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "tools"))
os.chdir(root)
print("Practical root:", root)

## Step 0 — Load the bundled labelled corpus

Everything is offline. The data is a small fictional set of financial sentences in
`data/labeled/headlines.csv`, two columns: `label` (1 = up, 0 = down) and `text`.
`load_dataset()` returns `(texts, labels)` as plain Python lists.

In [ ]:
import json
from collections import Counter
from tools._common import load_dataset, tokenize

texts, labels = load_dataset()
print(f"corpus size: {len(texts)} sentences")
print(f"label distribution: {dict(Counter(labels))}  (1 = up, 0 = down)")
print("\nfirst up (1) example:  ", texts[labels.index(1)])
print("first down (0) example:", texts[labels.index(0)])

## Step 1 — Tokenisation

The shared `tokenize()` helper lowercases, keeps content tokens of length >= 2, and
strips function/stop words. The sentiment-bearing words ("surged", "plunged") are
exactly what the classifier will lean on, so they survive tokenisation.

In [ ]:
sample = "Earnings beat expectations and revenue surged."
print("raw:   ", sample)
print("tokens:", tokenize(sample))

## Step 2 — Text → TF-IDF feature matrix

`TfidfVectorizer` (a tiny NumPy fit/transform implementation) learns a vocabulary and
IDF weights, then maps each sentence to an L2-normalised TF-IDF row. Crucially, the
vocabulary/IDF are learned from **training texts only** — here we fit on the full
corpus just to inspect its shape; the held-out evaluation later re-fits on the train
split so the test set never leaks into the features.

In [ ]:
import numpy as np
from tools.features import TfidfVectorizer

vec = TfidfVectorizer().fit(texts)
X = vec.transform(texts)
print(f"feature matrix shape: {X.shape}  (rows = sentences, cols = vocabulary)")
print(f"vocabulary size: {len(vec.vocabulary_)}")
print(f"sample vocabulary terms: {vec.vocabulary_[:12]}")
# every row is L2-normalised, so it reads as a direction in word space
print(f"row L2 norms all ~1.0: {np.allclose(np.linalg.norm(X, axis=1), 1.0)}")

### Try it — which words carry the most weight?

From the README's "things to try": inspect the TF-IDF vector of a single sentence to
see which words dominate. The sentiment terms should rise to the top.

In [ ]:
def show_active_features(sentence, vectorizer):
    row = vectorizer.transform([sentence])[0]
    active = sorted(
        ((vectorizer.vocabulary_[j], round(float(row[j]), 4)) for j in np.nonzero(row)[0]),
        key=lambda t: t[1], reverse=True,
    )
    print(f"input: {sentence!r}")
    print(f"non-zero features: {len(active)}")
    for word, weight in active:
        print(f"  {word:<14} {weight}")

show_active_features("Earnings beat expectations and revenue surged.", vec)

## Step 3 — Train the classifier and score single sentences

`train_full()` fits a vectoriser + a `LogisticRegressionGD` classifier on the whole
bundled dataset and returns `(model, vectorizer)` for ad-hoc prediction. The model
starts at zero weights and runs full-batch gradient descent — no scikit-learn, no
randomness. `classify_text()` returns `(label, P(up))` for one sentence.

In [ ]:
from tools.model import train_full, classify_text

model, full_vec = train_full()

for s in [
    "Earnings beat expectations and revenue surged to a record high.",
    "The stock plunged after profit missed and demand weakened.",
]:
    label, prob = classify_text(s, model, full_vec)
    print(f"P(up)={prob:.3f}  ->  {'UP (1)' if label == 1 else 'DOWN (0)'}   {s!r}")

### Try it — a deliberately neutral sentence sits near P(up) = 0.5

Another README suggestion: score a sentence with little directional text signal and
watch the probability hover around 0.5 — low confidence, as it should be.

In [ ]:
neutral = "The company held its annual meeting on Tuesday."
label, prob = classify_text(neutral, model, full_vec)
print(f"neutral sentence P(up) = {prob:.3f}  ({'UP (1)' if label == 1 else 'DOWN (0)'})")
print("(close to 0.5 = the model has little text signal to go on)")

## Step 4 — Held-out evaluation (the end-to-end step)

`run_pipeline()` is the full text-to-signal workflow: split the corpus with a fixed
seed, learn vocabulary/IDF on the **train half only**, fit the classifier, and score
the held-out half. The split is deterministic, so the accuracy and confusion counts
are reproducible.

In [ ]:
from tools.evaluate import run_pipeline

report = run_pipeline(test_frac=0.3, seed=42)
print(f"out-of-sample accuracy: {report['accuracy']:.3f} "
      f"({report['n_train']} train / {report['n_test']} test)")
print(json.dumps(report, indent=2))

### Try it — vary the seed and watch the held-out estimate move

The README's headline suggestion: one split is never the whole story. Re-running with
different seeds shifts which sentences land in the small test set, so the accuracy
estimate wobbles. That spread is exactly why you don't trust a single split.

In [ ]:
accs = []
for seed in [7, 21, 42, 99, 123]:
    r = run_pipeline(test_frac=0.3, seed=seed)
    accs.append(r["accuracy"])
    print(f"seed={seed:>3}: accuracy={r['accuracy']:.3f}  confusion={r['confusion']}")
print(f"\nrange over seeds: min={min(accs):.3f}  max={max(accs):.3f}  mean={np.mean(accs):.3f}")

## Recap / next steps

We ran the chapter's text-to-signal pipeline end to end on the real bundled data,
using the same deterministic reference tools the agentic project drives:

1. **Load** the labelled corpus — `tools._common.load_dataset` (60 sentences, 30/30).
2. **Tokenise** — `tools._common.tokenize` (lowercase, drop stop words).
3. **Featurise** — `tools.features.TfidfVectorizer` (fit/transform TF-IDF, L2-normalised).
4. **Train + classify** — `tools.model.train_full` / `classify_text`
   (`LogisticRegressionGD` by gradient descent).
5. **Evaluate** — `tools.evaluate.run_pipeline` (deterministic split, accuracy + confusion).

Things to explore next:
- Add new labelled rows to `data/labeled/headlines.csv` and re-run — does the
  vocabulary still separate the two classes?
- Sweep more seeds / `test_frac` values to map the full accuracy distribution.
- Inspect `show_active_features(...)` on your own sentences to see which words the
  linear model treats as signal.
- Run the offline test suite from the practical root: `python -m pytest -q`.